# DLsite 音声作品自动翻译

DLsite 日语音声作品 → 中日双语 SRT 字幕

**支持多音频文件，共用背景信息，各自独立翻译。**

**ASR 模式**：预设轴模式（基于参考 SRT 时间轴切片识别） / 直接模式（Whisper 自行分句）

**运行前确认**：菜单栏 → 修改 → 笔记本设置 → 硬件加速器选 T4 GPU

依次运行每个格子即可。

In [ ]:
# ① 克隆仓库 & 安装依赖
# 请将下方 URL 替换为你自己的仓库地址
!git clone https://github.com/ushi-C/DLtrans.git
%cd DLtrans
!pip install -r requirements.txt -q
!nvidia-smi

In [ ]:
# ② 导入模块 & 初始化
import sys
sys.path.insert(0, '/content/DLsiteAudioTranslator')

from pathlib import Path
from google.colab import files
from api_client import init_openai_client, usage_stats
from asr import run_asr, match_srt_to_audio
from background import AudioBackground
from config import ASR_MODE
from proofread import run_smart_proofread
from translate import run_parallel_translation
from utils import format_srt_time

client = init_openai_client()
print(f"ASR 模式: {ASR_MODE}")

In [ ]:
# ③ 输入背景信息（所有音频共用）
# 包括：作品标题、故事背景、登场角色
background = AudioBackground.input_interactive()

In [ ]:
# ========================================================
# 🚀 ④ 从谷歌网盘（Google Drive）拉取音频 & 参考 SRT 文件
# ========================================================
# 预设轴模式：请将音频文件和同名参考 SRT 一起放入网盘文件夹
# 例如：01_intro.wav ↔ 01_intro.srt
# ========================================================
from google.colab import drive
import os

# 1. 自动挂载网盘（如果已连接则自动跳过）
if not os.path.exists('/content/drive'):
    print("正在连接你的谷歌网盘，请在页面弹出的窗口中点击允许访问/连接...")
    drive.mount('/content/drive')
else:
    print("✅ 谷歌网盘已处于连接状态")

# 2. 设置你在网盘中存放音频的文件夹名称（纯英文）
FOLDER_NAME = 'audio_in'
drive_folder_path = f'/content/drive/MyDrive/{FOLDER_NAME}'

if not os.path.exists(drive_folder_path):
    os.makedirs(drive_folder_path)
    print(f"\n💡 提示：已自动创建【{FOLDER_NAME}】文件夹。")
    print(f"👉 请把音频文件和参考 SRT 文件拖进去，然后重新运行本单元格！")
    audio_files = []
    srt_mapping = {}
else:
    # 3. 扫描音频文件和 SRT 文件
    audio_exts = ('.mp3', '.m4a', '.wav', '.mp4', '.flac', '.aac', '.ogg')
    srt_exts = ('.srt',)

    audio_files = [
        os.path.join(drive_folder_path, f)
        for f in os.listdir(drive_folder_path)
        if f.lower().endswith(audio_exts)
    ]
    srt_files = [
        os.path.join(drive_folder_path, f)
        for f in os.listdir(drive_folder_path)
        if f.lower().endswith(srt_exts)
    ]

    # 4. 匹配 SRT 到音频（有 SRT 文件时按文件名匹配，否则在同目录自动查找）
    srt_mapping = match_srt_to_audio(audio_files, srt_files) if srt_files else match_srt_to_audio(audio_files)

    # 5. 打印结果
    print('\n' + '='*50)
    if audio_files:
        srt_count = len(srt_files) if srt_files else len(srt_mapping)
        print(f'✅ 共拉取到 {len(audio_files)} 个音频文件' + (f'，{srt_count} 个 SRT 文件' if srt_count else ''))
        print(f'   ASR 模式: {ASR_MODE}')
        print()
        for i, file_path in enumerate(audio_files, 1):
            fname = os.path.basename(file_path)
            srt = srt_mapping.get(file_path)
            if srt:
                print(f"   [{i}] {fname}  →  {os.path.basename(srt)}")
            else:
                tag = "⚠️ 无匹配 SRT" if ASR_MODE == "srt_preset" else "（直接模式）"
                print(f"   [{i}] {fname}  {tag}")
    else:
        print(f'⚠️ 文件夹为空，请先上传文件。')
    print('='*50)


In [ ]:
# ⑤ トラックリスト入力 → 正则拆轨 → 对齐到文件名 → 确认
#
# 【推奨】TRACKLIST 変数にペースト
# Colab の input() は複数行ペーストが不安定なため、変数代入を推奨。
# DLsite のトラックリストをそのままペーストしてください。
#
# 例：
# TRACKLIST = """
# 1.朝の挨拶 [06:48]
# 2.昼の挨拶 [05:30]
# 3.夜の挨拶 [10:00]
# 4.就寝 [08:15]
# 5.深夜の誘い [07:22]
# """
#
# 【備考】ファイル番号とリスト番号にズレがあっても自動的に順序で対齐します。
# 例: ファイル 4~8.mp3 / リスト 1~5 → 4→1, 5→2, ... と対齐

TRACKLIST = """
【ここにトラックリストをペースト】
"""

_raw = TRACKLIST.strip()
if _raw and "【ここに" not in _raw:
    track_descs = AudioBackground.apply_tracklist(_raw, audio_files)
    background.track_descriptions = track_descs
else:
    print("⚠️ TRACKLIST にトラックリストをペーストしてから再実行してください。")
    print("   または対話式入力を使用：")
    print("   track_descs = AudioBackground.input_track_descriptions(audio_files)")
    track_descs = {}
    background.track_descriptions = track_descs


In [ ]:
# ⑥ 逐个处理：ASR → 校对 → 翻译 → 写 SRT

srt_files_out = []

for audio_path in audio_files:
    filename = Path(audio_path).name
    srt_path = srt_mapping.get(audio_path)

    print(f"\n{'='*50}")
    print(f"🎬 处理: {filename}")
    if srt_path:
        print(f"   📝 参考 SRT: {Path(srt_path).name}")
    print(f"{'='*50}")

    # Step 1: ASR
    raw_asr = run_asr(audio_path, srt_path)
    if not raw_asr:
        print(f"⚠️ {filename} 未识别到任何内容，跳过")
        continue

    # Step 2: 校对（背景辅助）
    proofed_data = run_smart_proofread(client, raw_asr, background, filename)

    # Step 3: 翻译
    final_data = run_parallel_translation(client, proofed_data, background, filename)

    # Step 4: 写 SRT
    srt_file = f"{Path(audio_path).stem}_bilingual.srt"
    with open(srt_file, 'w', encoding='utf-8') as f:
        for i, s in enumerate(final_data, 1):
            f.write(
                f"{i}\n"
                f"{format_srt_time(s['start'])} --> {format_srt_time(s['end'])}\n"
                f"{s['ja']}\n{s['zh']}\n\n"
            )

    print(f"📄 已生成: {srt_file}")
    srt_files_out.append(srt_file)

# 汇总
print(f"\n{'='*50}")
print(f"✅ 全部完成！共生成 {len(srt_files_out)} 个 SRT 文件")
print(f"   Token 消耗估算: {usage_stats.total_tokens}")
print(f"{'='*50}")

# 逐个下载
for srt_file in srt_files_out:
    files.download(srt_file)
